# CMIP masked-point prediction — CERA-like architecture

### Data preprocessing

**Library import**

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
import torch
import torch.nn as nn
import json
from pathlib import Path
import matplotlib.pyplot as plt

EXPERIENCE 5 HYPERPARAMETERS     :

In [ ]:
setup_name = "cera"

In [ ]:
num_sample = 1000
chosen_autoencoder_type = "CNN" # choose between "MLP" and "CNN"
inv_alignment_method = "swd"  # "swd" for sliced Wasserstein distance, and "adversarial" for adversarial-classifier
mask_strategy = "central_6"  # options: central_1, central_6, checkerboard, hidden_bottom, keep_central_6, keep_central_1
val_fraction = 0.05
test_fraction = 0.15
cera_lambda_align = 0.0001
cera_lambda_pred = 0.01

In [ ]:
# ========================================
# AE Hyperparameters
# ========================================

ae_latent_dim = 64

# ========================================
# CERA Hyperparameters
# ========================================

cera_n_epochs = 100

cera_align_dims = 48
cera_eval_batch_size = 2048
cera_batch_size = 256  # Large batches help latent alignment stability.
cera_patience = 10
cera_learning_rate = 1e-3
cera_weight_decay = 1e-3
cera_n_projections = 64

**Masked-point prediction setup**

The target is now defined by hidden geographic point(s). The model input keeps all variables, but the hidden point(s) are filled with zeros so that the CNN input shape remains unchanged.


**Data Loading**

You have to run a PBS job to create the data loaded in the next cell. In the pbs file you can choose the following parameters :

- --max-abs-lat value \ (recommanded : 30)
- --patch-size-km value \ (recommanded : 1000)
- --time-stride value \ (recommanded : 24)
- --max-samples-per-climate value \ (recommanded : 1000)
- --random-seed value \ (recommanded : 42)

to run the PBS job use the following command in /glade/u/home/tsalin/CMIP: 

qsub run_build_multivariate_samples.pbs

to follow what's going on : 

qstat -u tsalin

In [ ]:
precomputed_dir = Path(f"/glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_NG_v{num_sample}")

if not precomputed_dir.exists():
    raise FileNotFoundError(f"Precomputed data directory not found: {precomputed_dir}")

with open(precomputed_dir / "run_config.json", "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

climate_order = list(run_cfg["climate_order"])
climate_colors = dict(run_cfg["climate_colors"])
selected_variables_full = list(run_cfg["selected_variables"])
max_abs_lat = float(run_cfg["max_abs_lat"])
patch_size_km = float(run_cfg["patch_size_km"])
time_stride = int(run_cfg["time_stride"])
max_samples_per_climate = int(run_cfg["max_samples_per_climate"])
random_seed = int(run_cfg["random_seed"])
n_lat = int(run_cfg["n_lat"])
n_lon = int(run_cfg["n_lon"])
grid_points_per_patch = int(run_cfg["grid_points_per_patch"])
n_patches = int(run_cfg["n_patches"])

features_by_climate_full = {
    c: np.load(precomputed_dir / f"features_{c}.npy")
    for c in climate_order
}
metadata_by_climate = {
    c: pd.read_csv(precomputed_dir / f"metadata_{c}.csv")
    for c in climate_order
}

n_variables_full = len(selected_variables_full)
expected_dim_full = n_variables_full * grid_points_per_patch

# Define the masked geographic point(s).

prediction_task = "masked_point_reconstruction"
masked_point_indices = None  # if None, it will be set to the central point after loading grid metadata
mask_fill_value = 0.0

if masked_point_indices is None:
    if mask_strategy == "central_1":
        masked_point_indices = np.array([
            4 * n_lon + 3,
        ], dtype=int)

    elif mask_strategy == "central_6":
        center_i = n_lat // 2
        center_j = n_lon // 2
        masked_point_indices = np.array([
            (center_i - 1) * n_lon + (center_j - 1),
            center_i * n_lon + (center_j - 1),
            (center_i - 1) * n_lon + center_j,
            center_i * n_lon + center_j,
            (center_i - 1) * n_lon + (center_j + 1),
            center_i * n_lon + (center_j + 1),
        ], dtype=int)

    elif mask_strategy == "checkerboard":
        masked_point_indices = np.array([
            i * n_lon + j
            for i in range(n_lat)
            for j in range(n_lon)
            if (i + j) % 2 == 0
        ], dtype=int)
    
    elif mask_strategy == "hidden_bottom":
        masked_point_indices = np.array([
            i * n_lon + j
            for i in range(n_lat)
            for j in range(n_lon)
            if i < n_lat // 2
        ], dtype=int)

    elif mask_strategy == "keep_central_6":
        visible_point_indices = np.array([
            5 * n_lon + 2,
            5 * n_lon + 3,
            5 * n_lon + 4,
            4 * n_lon + 2,
            4 * n_lon + 3,
            4 * n_lon + 4,
        ], dtype=int)
        masked_point_indices = np.setdiff1d(np.arange(grid_points_per_patch), visible_point_indices)
    
    elif mask_strategy == "keep_central_1":
        visible_point_indices = np.array([
            4 * n_lon + 3,
        ], dtype=int)
        masked_point_indices = np.setdiff1d(np.arange(grid_points_per_patch), visible_point_indices)

    else:
        raise ValueError(f"Unknown mask_strategy={mask_strategy!r}.")
else:
    masked_point_indices = np.asarray(masked_point_indices, dtype=int)

if masked_point_indices.ndim != 1 or masked_point_indices.size == 0:
    raise ValueError("masked_point_indices must be a non-empty 1D array of grid-point indices.")
if masked_point_indices.min() < 0 or masked_point_indices.max() >= grid_points_per_patch:
    raise ValueError(
        f"masked_point_indices must be between 0 and {grid_points_per_patch - 1}; "
        f"got {masked_point_indices.tolist()}."
    )

# Flattening convention of the precomputed samples is variable-major:
# [var0_point0, ..., var0_point69, var1_point0, ..., var15_point69].
masked_feature_columns = np.array(
    [var_idx * grid_points_per_patch + point_idx
     for var_idx in range(n_variables_full)
     for point_idx in masked_point_indices],
    dtype=int,
)

visible_feature_columns = np.setdiff1d(
    np.arange(expected_dim_full, dtype=int),
    masked_feature_columns,
    assume_unique=False,
)

masked_prediction_labels = [
    f"{var}@point_{point_idx}"
    for var in selected_variables_full
    for point_idx in masked_point_indices
]

# Input channels used by the CNN/MLP after preprocessing.
# The first 16 channels contain the standardized climate variables, with hidden
# point values replaced by mask_fill_value. The last channel is an explicit
# binary mask: 1 at hidden geographic points and 0 elsewhere.
mask_channel_name = "mask_hidden_point"
selected_variables = list(selected_variables_full) + [mask_channel_name]
input_value_dim = expected_dim_full
input_mask_dim = grid_points_per_patch
input_dim_with_mask = input_value_dim + input_mask_dim

# Raw placeholders; standardized masked inputs and standardized prediction labels
# are created in the rigorous preprocessing cell after the split has been built.
features_by_climate = {}
label_variable_by_climate_raw = {}
for c in climate_order:
    X_full = np.asarray(features_by_climate_full[c])
    if X_full.shape[1] != expected_dim_full:
        raise ValueError(
            f"Unexpected feature dimension for {c}: got {X_full.shape[1]}, expected {expected_dim_full}."
        )

    X_masked = X_full.copy()
    X_masked[:, masked_feature_columns] = mask_fill_value
    features_by_climate[c] = X_masked
    label_variable_by_climate_raw[c] = X_full[:, masked_feature_columns].copy()

sample_count_df = pd.read_csv(precomputed_dir / "sample_count.csv").set_index("scenario")
pre_sampling_df = pd.read_csv(precomputed_dir / "pre_sampling_df.csv").set_index("scenario")
patch_catalog = pd.read_csv(precomputed_dir / "patch_catalog.csv")
sampling_diagnostics_df = pd.read_csv(precomputed_dir / "sampling_diagnostics.csv").set_index("scenario")
nan_summary_by_variable_df = pd.read_csv(precomputed_dir / "nan_summary_by_variable.csv")
sample_pairs_by_climate = {
    c: pd.read_csv(precomputed_dir / f"sample_pairs_{c}.csv")
    for c in climate_order
}

display(sample_count_df)
display(pre_sampling_df)
display(sampling_diagnostics_df)
print(f"Prediction task: {prediction_task}")
print(f"Input channels used by the model: {selected_variables}")
print(f"Masked grid point indices: {masked_point_indices.tolist()}")
print(f"Masked feature columns: {masked_feature_columns.tolist()}")
print(f"Prediction target dimension: {len(masked_feature_columns)} = {n_variables_full} variables x {len(masked_point_indices)} hidden point(s)")
print(f"Prediction labels: {masked_prediction_labels}")


In [ ]:
# Backward-compatible aliases used later in the notebook
climate_order = list(climate_order)
climate_colors = dict(climate_colors)
sample_count_df = sample_count_df.copy()
pre_sampling_df = pre_sampling_df.copy()
patch_catalog = patch_catalog.copy()
sampling_diagnostics_df = sampling_diagnostics_df.copy()
nan_summary_by_variable_df = nan_summary_by_variable_df.copy()
cera_train_climates = ["historical", "ssp245"]

In [ ]:
# Predictor architecture: 5 hidden FC layers, 128 units, LeakyReLU.
cera_predictor_hidden_dim = 128
cera_predictor_n_hidden_layers = 5

# Adversarial classifier width (used only when inv_alignment_method == "adversarial").
cera_classifier_hidden_dims = (128, 64)

# Checkpoint: path to save/resume training state between PBS jobs
cera_checkpoint_path = Path("/glade/u/home/tsalin/CMIP/model_evaluation/CERA/cera_mask_checkpoint.pt")
cera_checkpoint_freq = 1  # Save checkpoint every N epochs (1 = every epoch)

if ae_latent_dim < cera_align_dims:
    raise ValueError(f"ae_latent_dim={ae_latent_dim} must be >= cera_align_dims={cera_align_dims}")

Patch visualization :

In [ ]:
def plot_patch_mask_from_catalog(masked_point_indices, patch_id=0, patch_catalog=patch_catalog):
    masked_point_indices = np.asarray(masked_point_indices, dtype=int)

    patch_info = patch_catalog.loc[patch_catalog["patch_id"] == patch_id].iloc[0]

    n_lat = int(patch_info["lat_stop_idx"] - patch_info["lat_start_idx"])
    n_lon = int(patch_info["lon_stop_idx"] - patch_info["lon_start_idx"])

    if n_lat * n_lon != grid_points_per_patch:
        raise ValueError(
            f"Patch shape mismatch: n_lat * n_lon = {n_lat * n_lon}, "
            f"but grid_points_per_patch = {grid_points_per_patch}"
        )

    lats = np.linspace(patch_info["lat_start"], patch_info["lat_stop"], n_lat)
    lons = np.linspace(patch_info["lon_start"], patch_info["lon_stop"], n_lon)

    lon_grid, lat_grid = np.meshgrid(lons, lats)

    flat_lons = lon_grid.ravel()
    flat_lats = lat_grid.ravel()

    is_masked = np.zeros(grid_points_per_patch, dtype=bool)
    is_masked[masked_point_indices] = True

    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)

    ax.scatter(
        flat_lons[~is_masked],
        flat_lats[~is_masked],
        s=70,
        color="#4C72B0",
        alpha=0.8,
        edgecolor="white",
        linewidth=0.6,
        label="Visible points",
        zorder=2,
    )

    ax.scatter(
        flat_lons[is_masked],
        flat_lats[is_masked],
        s=180,
        marker="X",
        color="#C44E52",
        edgecolor="black",
        linewidth=1.1,
        label="Masked points",
        zorder=5,
    )

    for idx in range(grid_points_per_patch):
        ax.text(
            flat_lons[idx],
            flat_lats[idx],
            str(idx),
            ha="center",
            va="center",
            fontsize=7,
            color="white" if is_masked[idx] else "black",
            fontweight="bold" if is_masked[idx] else "normal",
            zorder=6,
        )

    ax.set_title(f"Masked grid points for patch {patch_id}")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    ax.grid(alpha=0.25)
    ax.legend(frameon=True)

    ax.set_aspect("equal", adjustable="box")

    plt.show()


plot_patch_mask_from_catalog(masked_point_indices, patch_id=0)

**Latitude Band, 1000-km Patches, and Multivariate Samples**

This step builds one unified multivariate dataset used by all downstream analyses.

Workflow:
- select data inside +/- max_abs_lat (default: 30 deg);
- split the region into non-overlapping ~1000 km x 1000 km patches;
- for each climate, build samples from all variables and all lat/lon points inside each patch at sampled times;
- aggregate all climates to fit one common scaler

Standardization

In [ ]:
# Rigorous preprocessing for masked-point prediction
# ---------------------------------------------------
# Key choices:
# 1. Build train/val/test splits BEFORE standardization.
# 2. Fit normalization statistics only on training samples.
# 3. Do NOT use masked geographic points to fit normalization statistics.
# 4. Use a variable-wise scaler: for each climate variable, mean/std are estimated
#    from visible points only, then applied to all points of that variable.
# 5. Replace hidden point values by mask_fill_value AFTER standardization.
# 6. Add an explicit binary mask channel so the network can distinguish true
#    standardized zeros from deliberately hidden values.


def build_split_indices(data_by_climate, val_fraction=0.10, test_fraction=0.20, seed=42):
    """Build train/val/test split indices separately within each climate."""
    split_indices = {}
    rng = np.random.default_rng(seed)

    for climate, X in data_by_climate.items():
        n = X.shape[0]
        indices = np.arange(n)
        rng.shuffle(indices)

        n_test = max(1, int(round(test_fraction * n)))
        n_val = max(1, int(round(val_fraction * n)))
        n_train = max(1, n - n_val - n_test)

        train_idx = indices[:n_train]
        val_idx = indices[n_train:n_train + n_val]
        test_idx = indices[n_train + n_val:]

        split_indices[climate] = {
            "train": train_idx,
            "val": val_idx,
            "test": test_idx,
        }

    return split_indices

# Split raw data first, before any normalization.
ae_split_indices = build_split_indices(
    features_by_climate_full,
    val_fraction=val_fraction,
    test_fraction=test_fraction,
    seed=random_seed,
)

# ── RAM optimisation ───────
_eval_only_climates = [c for c in climate_order if c not in cera_train_climates]
if _eval_only_climates:
    for c in _eval_only_climates:
        _idx = ae_split_indices[c]["test"]
        features_by_climate_full[c]      = features_by_climate_full[c][_idx]
        features_by_climate[c]           = features_by_climate[c][_idx]
        label_variable_by_climate_raw[c] = label_variable_by_climate_raw[c][_idx]
        metadata_by_climate[c]           = metadata_by_climate[c].iloc[_idx].reset_index(drop=True)
        ae_split_indices[c] = {
            "train": np.array([], dtype=np.intp),
            "val":   np.array([], dtype=np.intp),
            "test":  np.arange(len(_idx), dtype=np.intp),
        }
    print(f"RAM opt : {_eval_only_climates} truncated to {len(_idx)} samples (test split).")
    del _idx
del _eval_only_climates
# ── End RAM optimisation ──────────────────────────────────────────────────────

visible_point_indices = np.setdiff1d(
    np.arange(grid_points_per_patch, dtype=int),
    masked_point_indices,
    assume_unique=False,
)
scaler_fit_climates = ["historical"]

scaler_fit_climates = [c for c in scaler_fit_climates if c in climate_order]
if not scaler_fit_climates:
    raise ValueError("scaler_fit_climates is empty after filtering against climate_order.")

# Estimate one mean/std per physical variable using only visible points from
# training samples of scaler_fit_climates. This avoids using hidden-point labels
# when defining the preprocessing transformation.
variable_means = np.zeros(n_variables_full, dtype=np.float64)
variable_stds = np.ones(n_variables_full, dtype=np.float64)

for var_idx, var_name in enumerate(selected_variables_full):
    visible_cols_for_var = var_idx * grid_points_per_patch + visible_point_indices
    train_visible_values = []

    for climate in scaler_fit_climates:
        train_idx = ae_split_indices[climate]["train"]
        X_train_raw = np.asarray(features_by_climate_full[climate][train_idx])
        train_visible_values.append(X_train_raw[:, visible_cols_for_var].reshape(-1))

    train_visible_values = np.concatenate(train_visible_values)
    finite_values = train_visible_values[np.isfinite(train_visible_values)]

    if finite_values.size == 0:
        raise ValueError(f"No finite visible training values found for variable {var_name!r}.")

    if var_name == "pr":
        finite_values = np.log1p(finite_values * 86400)

    mu = float(np.mean(finite_values))
    sigma = float(np.std(finite_values))
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1.0

    variable_means[var_idx] = mu
    variable_stds[var_idx] = sigma


# Apply variable-wise standardization to all samples and all points.
scaled_full_features_by_climate = {}
scaled_features_by_climate = {}
label_variable_by_climate = {}
mask_channel_by_climate = {}

mask_channel_template = np.zeros(grid_points_per_patch, dtype=np.float32)
mask_channel_template[masked_point_indices] = 1.0

for climate in climate_order:
    X_raw = np.asarray(features_by_climate_full[climate], dtype=np.float32)
    X_scaled_full = np.empty_like(X_raw, dtype=np.float32)

    for var_idx, var_name in enumerate(selected_variables_full):
        cols_for_var = slice(
            var_idx * grid_points_per_patch,
            (var_idx + 1) * grid_points_per_patch,
        )
        values = X_raw[:, cols_for_var]
        if var_name == "pr":
            values = np.log1p(values * 86400)
        X_scaled_full[:, cols_for_var] = (
            values - variable_means[var_idx]
        ) / variable_stds[var_idx]

    # Prediction target: true standardized values at masked points.
    y_masked = X_scaled_full[:, masked_feature_columns].copy()

    # Model input: standardized values with hidden point values replaced by 0.
    X_values_masked = X_scaled_full.copy()
    X_values_masked[:, masked_feature_columns] = mask_fill_value

    # Explicit mask channel: 1 at hidden point(s), 0 elsewhere.
    mask_channel = np.tile(mask_channel_template, (X_values_masked.shape[0], 1))
    X_model_input = np.concatenate([X_values_masked, mask_channel], axis=1).astype(np.float32)

    scaled_full_features_by_climate[climate] = X_scaled_full
    scaled_features_by_climate[climate] = X_model_input
    label_variable_by_climate[climate] = y_masked.astype(np.float32)
    mask_channel_by_climate[climate] = mask_channel

# Convenience aggregate preserving the same sample order as stacked climate features.
label_variable = np.vstack([label_variable_by_climate[c] for c in climate_order])

standardization_summary_df = pd.DataFrame({
    "variable": selected_variables_full,
    "mean_fit_on_train_visible_points": variable_means,
    "std_fit_on_train_visible_points": variable_stds,
})

print("Rigorous preprocessing completed.")
print(f"Scaler fit climates: {scaler_fit_climates}")
print(f"Masked point indices excluded from scaler fitting: {masked_point_indices.tolist()}")
print(f"Visible point count used for scaler fitting: {len(visible_point_indices)} / {grid_points_per_patch}")
print(f"Model input dimension: {input_dim_with_mask} = {expected_dim_full} standardized values + {grid_points_per_patch} mask values")
print("Scaled model input shapes:")
for climate in climate_order:
    print(
        f"  {climate}: X={scaled_features_by_climate[climate].shape}, "
        f"y_masked={label_variable_by_climate[climate].shape}"
    )

display(standardization_summary_df)


In [ ]:
# RAM Reduction
del features_by_climate_full
del features_by_climate
del label_variable_by_climate_raw

Utilities :

In [ ]:
torch.manual_seed(random_seed)
np.random.seed(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Train/test split

In [ ]:
# Train/validation/test split
# The split has already been built in the rigorous preprocessing cell, before standardization.
# We keep this cell for compatibility with the rest of the notebook.

if "ae_split_indices" not in globals():
    raise RuntimeError("ae_split_indices is missing. Run the rigorous preprocessing cell first.")

split_summary_rows = []
for climate in climate_order:
    split_summary_rows.append({
        "scenario": climate,
        "n_train": len(ae_split_indices[climate]["train"]),
        "n_val": len(ae_split_indices[climate]["val"]),
        "n_test": len(ae_split_indices[climate]["test"]),
    })

split_summary_df = pd.DataFrame(split_summary_rows)
display(split_summary_df)


Structure inspired by CERA (CNN2D) - it uses the spatial structure of the samples

In [ ]:


class CNNEncoder(nn.Module):
    def __init__(self, input_channels, spatial_shape, latent_dim):
        super().__init__()
        self.input_channels = input_channels
        self.spatial_shape = spatial_shape
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, input_channels, *spatial_shape)
            feature_map = self.features(dummy)
            self.feature_shape = tuple(feature_map.shape[1:])
            self.flatten_dim = int(np.prod(self.feature_shape))
        self.projection = nn.Linear(self.flatten_dim, latent_dim)

    def forward(self, x):
        if x.ndim == 2:
            x = x.reshape(x.shape[0], self.input_channels, *self.spatial_shape)
        elif x.ndim != 4:
            raise ValueError("Expected a 2D flat batch or a 4D image batch.")
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        return self.projection(x)


class CNNDecoder(nn.Module):
    def __init__(self, output_channels, spatial_shape, latent_dim, feature_shape):
        super().__init__()
        self.output_channels = output_channels
        self.spatial_shape = spatial_shape
        self.feature_shape = feature_shape
        self.project = nn.Sequential(
            nn.Linear(latent_dim, int(np.prod(feature_shape))),
            nn.ReLU(inplace=True),
        )
        self.refine = nn.Sequential(
            nn.Conv2d(feature_shape[0], 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(size=spatial_shape, mode="bilinear", align_corners=False),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, output_channels, kernel_size=3, padding=1),
        )

    def forward(self, z):
        x = self.project(z)
        x = x.reshape(z.shape[0], *self.feature_shape)
        x = self.refine(x)
        return torch.flatten(x, start_dim=1)


class CNNAutoEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dims=None):
        super().__init__()
        _ = hidden_dims
        self.input_channels = len(selected_variables)
        self.spatial_shape = (n_lat, n_lon)

        expected_dim = self.input_channels * grid_points_per_patch
        if input_dim != expected_dim:
            raise ValueError(
                f"input_dim={input_dim} is incompatible with a CNN reshape using "
                f"{self.input_channels} channels and {grid_points_per_patch} grid points per patch."
            )

        self.encoder = CNNEncoder(self.input_channels, self.spatial_shape, latent_dim)
        self.decoder = CNNDecoder(self.input_channels, self.spatial_shape, latent_dim, self.encoder.feature_shape)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

A more classical structure using only MLPs - it doesn't use the spatial structure but apparently it performs better

In [ ]:
input_dim = next(iter(scaled_features_by_climate.values())).shape[1]
MLP_hidden_dim_1 = max(256, min(1024, input_dim // 2))
MLP_hidden_dim_2 = max(128, min(512, input_dim // 8))


class MLPEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dims=(512, 256)):
        super().__init__()
        h1, h2 = hidden_dims
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, latent_dim),
        )

    def forward(self, x):
        return self.net(x)


class MLPDecoder(nn.Module):
    def __init__(self, latent_dim, output_dim, hidden_dims=(256, 512)):
        super().__init__()
        h1, h2 = hidden_dims
        self.net = nn.Sequential(
            nn.Linear(latent_dim, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, output_dim),
        )

    def forward(self, z):
        return self.net(z)


class MLPAutoEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dims=(512, 256)):
        super().__init__()
        self.encoder = MLPEncoder(input_dim, latent_dim, hidden_dims=hidden_dims)
        self.decoder = MLPDecoder(latent_dim, input_dim, hidden_dims=hidden_dims[::-1])

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z


Utilities

In [ ]:
def visible_reconstruction_loss(x_hat: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """MSE reconstruction loss restricted to original non-masked climate features.

    The explicit mask channel is part of the model input, but it is not a reconstruction target.
    Hidden point values are also excluded from this loss; they are handled only by
    the prediction loss.
    """
    return nn.functional.mse_loss(
        x_hat[:, visible_feature_columns.tolist()],
        x[:, visible_feature_columns.tolist()],
    )


General architecture of the AE :

In [ ]:
def _build_autoencoder_by_type(autoencoder_type: str, input_dim: int, latent_dim: int):
    if autoencoder_type == "CNN":
        return CNNAutoEncoder(
            input_dim=input_dim,
            latent_dim=latent_dim,
        )
    if autoencoder_type == "MLP":
        return MLPAutoEncoder(
            input_dim=input_dim,
            latent_dim=latent_dim,
            hidden_dims=(MLP_hidden_dim_1, MLP_hidden_dim_2),
        )
    raise ValueError(f"Unsupported autoencoder type: {autoencoder_type}")

## Fifth Experiment - CERA-like architecture

In this fifth experiment, we add a predictor to the architecture considered in the fourth experiment. We thus now consider : AE (constructed either with cnn2D or MLPs) (and with either sliced wasserstein distance alignment or adversarial classifier alignment) and a predictor using only the aligned part of the historical climate latent representations. The predictor needs to predict the temperature field (tas) over the whole grid of the samples. This architecture will be called a CERA-like architecture.

The same test/train/val split than before is used.

We start by training the CERA-like architecture.

### Training a predictive invariant AE (historical + ssp245) - CERA like architecture

This CERA-like setup extends the invariant AE by adding a masked-point predictor:

AE input: multivariate samples from historical + ssp245 climates, with the masked geographic point(s) filled by mask_fill_value and an explicit binary mask channel appended.
AE losses: reconstruction (on visible, non-masked points only) + latent alignment on the first align_dims (48) latent dimensions.
Predictor: MLP on the aligned latent dimensions (historical batch only) to predict all variables at the masked geographic point(s).
Global loss used for AE update:
$$
L_{\text{total}} = (1 - \lambda_{\text{pred}} - \lambda_{\text{align}}) \cdot L_{\text{rec}} + \lambda_{\text{align}} \cdot L_{\text{align}} + \lambda_{\text{pred}} \cdot L_{\text{pred}}
$$

And we perform the predictor update at the same time, to be consistent with the end-to-end training used in the original CERA architecture.

Architecture - predictor part :

In [ ]:
# CERA-like predictive invariant AE: Classes and Helper Functions

class CERAPredictor(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int = 128, n_hidden_layers: int = 5):
        super().__init__()
        layers = []
        in_dim = input_dim
        for _ in range(n_hidden_layers):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.LeakyReLU(negative_slope=0.1))
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class CERAGradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


def cera_gradient_reversal(x: torch.Tensor, lambd: float = 1.0) -> torch.Tensor:
    return CERAGradientReversalFunction.apply(x, lambd)


class CERALatentDomainClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dims=(128, 64), n_domains: int = 2):
        super().__init__()
        h1, h2 = hidden_dims
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, n_domains),
        )

    def forward(self, z: torch.Tensor, grl_lambda: float = 1.0) -> torch.Tensor:
        return self.net(cera_gradient_reversal(z, grl_lambda))


def cera_swd_alignment_loss_torch(
    z_hist: torch.Tensor,
    z_ssp: torch.Tensor,
    n_projections: int = 64,
    eps: float = 1e-12,
) -> torch.Tensor:
    d = z_hist.shape[1]
    directions = torch.randn(n_projections, d, device=z_hist.device, dtype=z_hist.dtype)
    directions = directions / (torch.norm(directions, dim=1, keepdim=True) + eps)

    proj_hist = z_hist @ directions.T
    proj_ssp = z_ssp @ directions.T

    proj_hist_sorted, _ = torch.sort(proj_hist, dim=0)
    proj_ssp_sorted, _ = torch.sort(proj_ssp, dim=0)

    m = min(proj_hist_sorted.shape[0], proj_ssp_sorted.shape[0])
    if m == 0:
        return torch.tensor(0.0, device=z_hist.device, dtype=z_hist.dtype)

    return torch.mean(torch.abs(proj_hist_sorted[:m] - proj_ssp_sorted[:m]))


def cera_make_labeled_loader(
    data_by_climate,
    labels_by_climate,
    split_indices,
    climate: str,
    split: str,
    batch_size: int,
    shuffle: bool,
    drop_last: bool,
 ):
    idx = split_indices[climate][split]
    X = np.asarray(data_by_climate[climate][idx], dtype=np.float32)
    y = np.asarray(labels_by_climate[climate][idx], dtype=np.float32)
    ds = torch.utils.data.TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)

Full training function :

In [ ]:
def train_predictive_invariant_autoencoder(
    alignment_method: str = "adversarial",
    train_climates=None,
    latent_dim: int = 64,
    n_epochs: int = 40,
    lr: float = 1e-3,
    batch_size: int = 1024,
    weight_decay: float = 1e-5,
    align_dims: int = 48,
    lambda_align: float = 0.2,
    lambda_pred: float = 1.0,
    n_projections: int = 64,
    predictor_hidden_dim: int = 128,
    predictor_n_hidden_layers: int = 5,
    classifier_hidden_dims=(128, 64),
    checkpoint_path=None,
    checkpoint_freq=1,
 ):
    """
    Train CERA-like predictive invariant autoencoder with joint end-to-end optimization.
    
    Single backward pass and optimizer step per batch:
    L_total = (1 - lambda_align - lambda_pred) * L_rec + lambda_align * L_align + lambda_pred * L_pred
    
    - Predictor takes only aligned latent dimensions as input.
    - L_rec uses only visible, non-masked climate features from both historical and warm batches.
    - L_align uses aligned latent parts of both batches.
    - L_pred uses predictor output and the masked point values as targets.
    - Predictor parameters are NOT frozen during AE updates.
    """
    if train_climates is None:
        train_climates = ["historical", "ssp245"]
    if train_climates != ["historical", "ssp245"]:
        raise ValueError("This setup expects train_climates=['historical', 'ssp245']")
    if alignment_method not in {"swd", "adversarial"}:
        raise ValueError("alignment_method must be either 'swd' or 'adversarial'.")

    input_dim_local = next(iter(scaled_features_by_climate.values())).shape[1]
    output_dim_pred = int(len(masked_feature_columns))

    model = _build_autoencoder_by_type(chosen_autoencoder_type, input_dim_local, latent_dim).to(device)
    predictor = CERAPredictor(
        input_dim=align_dims,
        output_dim=output_dim_pred,
        hidden_dim=predictor_hidden_dim,
        n_hidden_layers=predictor_n_hidden_layers,
    ).to(device)

    domain_classifier = None
    if alignment_method == "adversarial":
        domain_classifier = CERALatentDomainClassifier(
            input_dim=align_dims,
            hidden_dims=classifier_hidden_dims,
        ).to(device)

    # Single joint optimizer for AE + predictor + domain_classifier
    ae_and_pred_params = list(model.parameters()) + list(predictor.parameters())
    if domain_classifier is not None:
        ae_and_pred_params += list(domain_classifier.parameters())
    optimizer = torch.optim.AdamW(ae_and_pred_params, lr=lr, weight_decay=weight_decay)

    mse_loss_fn = nn.MSELoss()
    ce_loss_fn = nn.CrossEntropyLoss()

    half_batch = max(2, batch_size // 2)
    train_loader_hist = cera_make_labeled_loader(
        scaled_features_by_climate, label_variable_by_climate, ae_split_indices,
        "historical", "train", half_batch, True, True
    )
    train_loader_ssp = cera_make_labeled_loader(
        scaled_features_by_climate, label_variable_by_climate, ae_split_indices,
        "ssp245", "train", half_batch, True, True
    )
    val_loader_hist = cera_make_labeled_loader(
        scaled_features_by_climate, label_variable_by_climate, ae_split_indices,
        "historical", "val", half_batch, False, True
    )
    val_loader_ssp = cera_make_labeled_loader(
        scaled_features_by_climate, label_variable_by_climate, ae_split_indices,
        "ssp245", "val", half_batch, False, True
    )

    best_val = np.inf
    best_epoch = -1
    patience_counter = 0
    best_model_state = None
    best_pred_state = None
    best_domain_state = None
    history = []

    start_epoch = 1
    if checkpoint_path is not None:
        _ckpt_path = Path(checkpoint_path)
        if _ckpt_path.exists():
            _ckpt = torch.load(_ckpt_path, map_location=device)
            model.load_state_dict(_ckpt["model_state"])
            predictor.load_state_dict(_ckpt["predictor_state"])
            optimizer.load_state_dict(_ckpt["optimizer_state"])
            if domain_classifier is not None and _ckpt.get("domain_classifier_state"):
                domain_classifier.load_state_dict(_ckpt["domain_classifier_state"])
            start_epoch       = _ckpt["epoch"] + 1
            best_val          = _ckpt["best_val"]
            best_epoch        = _ckpt["best_epoch"]
            patience_counter  = _ckpt["patience_counter"]
            best_model_state  = _ckpt["best_model_state"]
            best_pred_state   = _ckpt["best_pred_state"]
            best_domain_state = _ckpt.get("best_domain_state")
            history           = _ckpt["history"]
            print(f"[CHECKPOINT] Resume from epoch {start_epoch} "
                  f"(best: epoch {best_epoch}, val={best_val:.6f})")

    for epoch in range(start_epoch, n_epochs + 1):
        model.train()
        predictor.train()
        if domain_classifier is not None:
            domain_classifier.train()

        train_recon_losses = []
        train_align_losses = []
        train_pred_losses = []
        train_total_losses = []

        n_steps = min(len(train_loader_hist), len(train_loader_ssp))
        hist_iter = iter(train_loader_hist)
        ssp_iter = iter(train_loader_ssp)

        for _ in range(n_steps):
            xb_hist, yb_hist = next(hist_iter)
            xb_ssp, _ = next(ssp_iter)
            xb_hist = xb_hist.to(device)
            yb_hist = yb_hist.to(device)
            xb_ssp = xb_ssp.to(device)

            # Joint end-to-end optimization: single backward pass, single optimizer step
            optimizer.zero_grad()

            # Forward pass on concatenated batch (both historical and ssp)
            xb = torch.cat([xb_hist, xb_ssp], dim=0)
            x_hat, z = model(xb)
            
            # L_rec: reconstruction loss only on visible, non-masked climate features
            recon_loss = visible_reconstruction_loss(x_hat, xb)

            # Extract aligned latent dimensions for both climates
            z_hist = z[: xb_hist.shape[0], :align_dims]
            z_ssp = z[xb_hist.shape[0] :, :align_dims]

            # L_align: alignment loss between aligned dimensions
            if alignment_method == "swd":
                align_loss = cera_swd_alignment_loss_torch(
                    z_hist, z_ssp, n_projections=n_projections
                )
            else:
                domain_targets = torch.cat([
                    torch.zeros(xb_hist.shape[0], dtype=torch.long, device=device),
                    torch.ones(xb_ssp.shape[0], dtype=torch.long, device=device),
                ], dim=0)
                z_align = torch.cat([z_hist, z_ssp], dim=0)
                domain_logits = domain_classifier(z_align, grl_lambda=1.0)
                align_loss = ce_loss_fn(domain_logits, domain_targets)

            # L_pred: masked-point prediction loss (only on historical batch)
            y_pred_hist = predictor(z_hist)
            pred_loss = mse_loss_fn(y_pred_hist, yb_hist)

            # Total loss: joint combination of all three losses
            total_loss = (1 - lambda_align - lambda_pred) * recon_loss + lambda_align * align_loss + lambda_pred * pred_loss
            
            # Single backward pass
            total_loss.backward()
            
            # Single optimizer step
            optimizer.step()

            train_recon_losses.append(float(recon_loss.detach().cpu().item()))
            train_align_losses.append(float(align_loss.detach().cpu().item()))
            train_pred_losses.append(float(pred_loss.detach().cpu().item()))
            train_total_losses.append(float(total_loss.detach().cpu().item()))

        model.eval()
        predictor.eval()
        if domain_classifier is not None:
            domain_classifier.eval()

        val_recon_losses = []
        val_align_losses = []
        val_pred_losses = []
        val_total_losses = []

        with torch.no_grad():
            n_val_steps = min(len(val_loader_hist), len(val_loader_ssp))
            hist_val_iter = iter(val_loader_hist)
            ssp_val_iter = iter(val_loader_ssp)

            for _ in range(n_val_steps):
                xb_hist, yb_hist = next(hist_val_iter)
                xb_ssp, _ = next(ssp_val_iter)
                xb_hist = xb_hist.to(device)
                yb_hist = yb_hist.to(device)
                xb_ssp = xb_ssp.to(device)

                xb = torch.cat([xb_hist, xb_ssp], dim=0)
                x_hat, z = model(xb)
                recon_loss = visible_reconstruction_loss(x_hat, xb)

                z_hist = z[: xb_hist.shape[0], :align_dims]
                z_ssp = z[xb_hist.shape[0] :, :align_dims]

                if alignment_method == "swd":
                    align_loss = cera_swd_alignment_loss_torch(
                        z_hist, z_ssp, n_projections=n_projections
                    )
                else:
                    domain_targets = torch.cat([
                        torch.zeros(xb_hist.shape[0], dtype=torch.long, device=device),
                        torch.ones(xb_ssp.shape[0], dtype=torch.long, device=device),
                    ], dim=0)
                    z_align = torch.cat([z_hist, z_ssp], dim=0)
                    domain_logits = domain_classifier(z_align, grl_lambda=1.0)
                    align_loss = ce_loss_fn(domain_logits, domain_targets)

                y_pred_hist = predictor(z_hist)
                pred_loss = mse_loss_fn(y_pred_hist, yb_hist)
                total_loss = (1 - lambda_align - lambda_pred) * recon_loss + lambda_align * align_loss + lambda_pred * pred_loss

                val_recon_losses.append(float(recon_loss.detach().cpu().item()))
                val_align_losses.append(float(align_loss.detach().cpu().item()))
                val_pred_losses.append(float(pred_loss.detach().cpu().item()))
                val_total_losses.append(float(total_loss.detach().cpu().item()))

        train_recon = float(np.mean(train_recon_losses)) if train_recon_losses else np.nan
        train_align = float(np.mean(train_align_losses)) if train_align_losses else np.nan
        train_pred = float(np.mean(train_pred_losses)) if train_pred_losses else np.nan
        train_total = float(np.mean(train_total_losses)) if train_total_losses else np.nan

        val_recon = float(np.mean(val_recon_losses)) if val_recon_losses else np.nan
        val_align = float(np.mean(val_align_losses)) if val_align_losses else np.nan
        val_pred = float(np.mean(val_pred_losses)) if val_pred_losses else np.nan
        val_total = float(np.mean(val_total_losses)) if val_total_losses else np.nan

        history.append(
            {
                "epoch": epoch,
                "train_recon_loss": train_recon,
                "train_align_loss": train_align,
                "train_pred_loss": train_pred,
                "train_total_loss": train_total,
                "val_recon_loss": val_recon,
                "val_align_loss": val_align,
                "val_pred_loss": val_pred,
                "val_total_loss": val_total,
            }
        )

        if checkpoint_path is not None and epoch % checkpoint_freq == 0:
            torch.save({
                "epoch": epoch,
                "model_state": {k: v.cpu().clone() for k, v in model.state_dict().items()},
                "predictor_state": {k: v.cpu().clone() for k, v in predictor.state_dict().items()},
                "optimizer_state": optimizer.state_dict(),
                "domain_classifier_state": (
                    {k: v.cpu().clone() for k, v in domain_classifier.state_dict().items()}
                    if domain_classifier is not None else None
                ),
                "best_val": best_val,
                "best_epoch": best_epoch,
                "patience_counter": patience_counter,
                "best_model_state": best_model_state,
                "best_pred_state": best_pred_state,
                "best_domain_state": best_domain_state,
                "history": history,
            }, checkpoint_path)

        if val_total < best_val:
            best_val = val_total
            best_epoch = epoch
            best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_pred_state = {k: v.detach().cpu().clone() for k, v in predictor.state_dict().items()}
            if domain_classifier is not None:
                best_domain_state = {
                    k: v.detach().cpu().clone() for k, v in domain_classifier.state_dict().items()
                }
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= cera_patience:
                if checkpoint_path is not None:
                    torch.save({
                        "epoch": epoch,
                        "model_state": {k: v.cpu().clone() for k, v in model.state_dict().items()},
                        "predictor_state": {k: v.cpu().clone() for k, v in predictor.state_dict().items()},
                        "optimizer_state": optimizer.state_dict(),
                        "domain_classifier_state": (
                            {k: v.cpu().clone() for k, v in domain_classifier.state_dict().items()}
                            if domain_classifier is not None else None
                        ),
                        "best_val": best_val,
                        "best_epoch": best_epoch,
                        "patience_counter": patience_counter,
                        "best_model_state": best_model_state,
                        "best_pred_state": best_pred_state,
                        "best_domain_state": best_domain_state,
                        "history": history,
                    }, checkpoint_path)
                break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    if best_pred_state is not None:
        predictor.load_state_dict(best_pred_state)
    if (domain_classifier is not None) and (best_domain_state is not None):
        domain_classifier.load_state_dict(best_domain_state)

    history_df = pd.DataFrame(history)
    return model, predictor, history_df, best_epoch, domain_classifier


training :

In [ ]:
cera_ae_model, cera_predictor, cera_history_df, cera_best_epoch, cera_domain_classifier = train_predictive_invariant_autoencoder(
    alignment_method=inv_alignment_method,
    train_climates=cera_train_climates,
    latent_dim=ae_latent_dim,
    n_epochs=cera_n_epochs,
    lr=cera_learning_rate,
    batch_size=cera_batch_size,
    weight_decay=cera_weight_decay,
    align_dims=cera_align_dims,
    lambda_align=cera_lambda_align,
    lambda_pred=cera_lambda_pred,
    n_projections=cera_n_projections,
    predictor_hidden_dim=cera_predictor_hidden_dim,
    predictor_n_hidden_layers=cera_predictor_n_hidden_layers,
    classifier_hidden_dims=cera_classifier_hidden_dims,
    checkpoint_path=cera_checkpoint_path,
    checkpoint_freq=cera_checkpoint_freq,
 )

print("Chosen autoencoder type:", chosen_autoencoder_type)
print("CERA alignment method:", inv_alignment_method)
print("CERA training climates:", ", ".join(cera_train_climates))
print("Best epoch (predictive invariant AE):", cera_best_epoch)

display(cera_history_df.tail())

### Reconstruction and masked-point prediction quality on test data - EXTRACTION

We evaluate:
- AE reconstruction quality only on **visible, non-masked climate features**.
- masked-point prediction quality only on the **hidden geographic point(s)**.

The explicit binary mask channel is used as an input to the model, but it is not used as a reconstruction target.

Reminder: predictor training uses the historical train split for the prediction loss, while alignment uses historical and SSP245.


In [ ]:
component_order = {"reconstruction": 0, "prediction": 1}


# ---------------------------------------------------------------------
# Visible / masked reconstruction metadata
# ---------------------------------------------------------------------

visible_point_indices = np.setdiff1d(
    np.arange(grid_points_per_patch, dtype=int),
    masked_point_indices,
    assume_unique=False,
)

visible_feature_columns = np.setdiff1d(
    np.arange(expected_dim_full, dtype=int),
    masked_feature_columns,
    assume_unique=False,
)

reconstruction_labels_visible = [
    f"{var}@point_{point_idx}"
    for var in selected_variables_full
    for point_idx in visible_point_indices
]


def denormalize_full_features(
    X_normalized: np.ndarray,
    variable_means: np.ndarray,
    variable_stds: np.ndarray,
    n_points: int,
    n_variables: int,
) -> np.ndarray:
    X_denorm = np.asarray(X_normalized, dtype=np.float32).copy()

    for var_idx, var_name in enumerate(selected_variables_full):
        cols_for_var = slice(var_idx * n_points, (var_idx + 1) * n_points)
        values = X_denorm[:, cols_for_var] * variable_stds[var_idx] + variable_means[var_idx]
        if var_name == "pr":
            values = np.expm1(values)
        X_denorm[:, cols_for_var] = values

    return X_denorm


def denormalize_masked_targets(
    y_normalized: np.ndarray,
    variable_means: np.ndarray,
    variable_stds: np.ndarray,
    n_masked_points: int,
    n_variables: int,
) -> np.ndarray:
    y_denorm = np.asarray(y_normalized, dtype=np.float32).copy()

    for var_idx, var_name in enumerate(selected_variables_full):
        start_col = var_idx * n_masked_points
        end_col = (var_idx + 1) * n_masked_points
        values = y_denorm[:, start_col:end_col] * variable_stds[var_idx] + variable_means[var_idx]
        if var_name == "pr":
            values = np.expm1(values)
        y_denorm[:, start_col:end_col] = values

    return y_denorm


def _build_meta_df(component, climate, metadata, latent_dim, pred_latent_dim, pred_task_val=None):
    df = metadata.copy().reset_index(drop=True)
    df = df.drop(columns=["scenario"], errors="ignore")
    df.insert(0, "experiment", "CMIP_mask_exp5_CERA")
    df.insert(1, "component", component)
    df.insert(2, "component_order", component_order[component])
    df.insert(3, "scenario", climate)
    df.insert(4, "scenario_order", int(climate_order.index(climate)))
    df.insert(5, "sample_idx", np.arange(len(df)))
    df.insert(6, "latent_dim", int(latent_dim))
    df.insert(7, "prediction_latent_dim", int(pred_latent_dim))
    if pred_task_val is not None:
        df.insert(8, "prediction_task", pred_task_val)
    return df


def _predict_cera_in_batches(model, predictor, X, batch_size=4096):
    x_hat_chunks = []
    y_hat_chunks = []
    z_chunks = []

    model.eval()
    predictor.eval()

    with torch.inference_mode():
        for start in range(0, X.shape[0], batch_size):
            xb = torch.as_tensor(
                X[start:start + batch_size],
                dtype=torch.float32,
                device=device,
            )

            x_hat, z = model(xb)
            y_pred = predictor(z[:, :cera_align_dims])

            x_hat_chunks.append(x_hat.detach().cpu().numpy())
            y_hat_chunks.append(y_pred.detach().cpu().numpy())
            z_chunks.append(z.detach().cpu().numpy())

    return (
        np.vstack(x_hat_chunks),
        np.vstack(y_hat_chunks),
        np.vstack(z_chunks),
    )


# Accumulators
meta_dfs_recon, meta_dfs_pred = [], []
truth_arrays_recon, pred_arrays_recon = [], []
truth_arrays_pred, pred_arrays_pred = [], []

n_masked_points = int(len(masked_point_indices))

for climate in climate_order:
    idx = ae_split_indices[climate]["test"]

    X = np.asarray(
        scaled_features_by_climate[climate][idx],
        dtype=np.float32,
    )

    y_true_scaled = np.asarray(
        label_variable_by_climate[climate][idx],
        dtype=np.float32,
    )

    metadata = metadata_by_climate[climate].iloc[idx].reset_index(drop=True)

    X_hat_scaled, y_hat_scaled, z_np = _predict_cera_in_batches(
        cera_ae_model,
        cera_predictor,
        X,
        batch_size=4096,
    )

    # -----------------------------------------------------------------
    # Reconstruction: export only visible features
    # -----------------------------------------------------------------
    X_true_scaled = np.asarray(
        scaled_full_features_by_climate[climate][idx],
        dtype=np.float32,
    )

    X_true_physical = denormalize_full_features(
        X_true_scaled,
        variable_means,
        variable_stds,
        n_points=grid_points_per_patch,
        n_variables=n_variables_full,
    )

    X_hat_physical = denormalize_full_features(
        X_hat_scaled,
        variable_means,
        variable_stds,
        n_points=grid_points_per_patch,
        n_variables=n_variables_full,
    )

    meta_dfs_recon.append(
        _build_meta_df("reconstruction", climate, metadata, z_np.shape[1], cera_align_dims)
    )
    truth_arrays_recon.append(X_true_physical[:, visible_feature_columns].astype(np.float32))
    pred_arrays_recon.append(X_hat_physical[:, visible_feature_columns].astype(np.float32))

    # -----------------------------------------------------------------
    # Prediction: export masked-point values
    # -----------------------------------------------------------------
    y_true_physical = denormalize_masked_targets(
        y_true_scaled,
        variable_means,
        variable_stds,
        n_masked_points=n_masked_points,
        n_variables=n_variables_full,
    )

    y_hat_physical = denormalize_masked_targets(
        y_hat_scaled,
        variable_means,
        variable_stds,
        n_masked_points=n_masked_points,
        n_variables=n_variables_full,
    )

    meta_dfs_pred.append(
        _build_meta_df("prediction", climate, metadata, z_np.shape[1], cera_align_dims, pred_task_val=prediction_task)
    )
    truth_arrays_pred.append(y_true_physical.astype(np.float32))
    pred_arrays_pred.append(y_hat_physical.astype(np.float32))


cera_quality_payload = {
    "meta_reconstruction": pd.concat(meta_dfs_recon, ignore_index=True),
    "meta_prediction": pd.concat(meta_dfs_pred, ignore_index=True),
    "truth_reconstruction": np.concatenate(truth_arrays_recon, axis=0),
    "pred_reconstruction": np.concatenate(pred_arrays_recon, axis=0),
    "truth_prediction": np.concatenate(truth_arrays_pred, axis=0),
    "pred_prediction": np.concatenate(pred_arrays_pred, axis=0),
    "reconstruction_value_names": reconstruction_labels_visible,
    "prediction_value_names": masked_prediction_labels,
    "masked_point_indices": masked_point_indices.tolist(),
    "masked_feature_columns": masked_feature_columns.tolist(),
    "visible_point_indices": visible_point_indices.tolist(),
    "visible_feature_columns": visible_feature_columns.tolist(),
}

display(cera_quality_payload["meta_reconstruction"].head())
display(cera_quality_payload["meta_prediction"].head())
display(cera_quality_payload["truth_reconstruction"].shape)
display(cera_quality_payload["truth_prediction"].shape)


### Extracting representations of CERA-like latent space

In [ ]:
# Extract latent test representations for all climates.
cera_latent_by_climate = {}
cera_latent_test_metadata_by_climate = {}

cera_ae_model.eval()
with torch.no_grad():
    for climate in climate_order:
        idx = ae_split_indices[climate]["test"]
        X = np.asarray(scaled_features_by_climate[climate][idx], dtype=np.float32)

        z_chunks = []
        for start in range(0, X.shape[0], 4096):
            xb = torch.tensor(X[start:start + 4096], dtype=torch.float32, device=device)
            _, z = cera_ae_model(xb)
            z_chunks.append(z.detach().cpu().numpy())

        cera_latent_by_climate[climate] = np.vstack(z_chunks)
        cera_latent_test_metadata_by_climate[climate] = metadata_by_climate[climate].iloc[idx].reset_index(drop=True)


if cera_align_dims >= ae_latent_dim:
    raise ValueError(
        f"Cannot run non-aligned PCA because cera_align_dims={cera_align_dims} and ae_latent_dim={ae_latent_dim}."
    )


In [ ]:
# Delete the checkpoint once that all the notebook has run successfully
if cera_checkpoint_path.exists():
    cera_checkpoint_path.unlink()
    print(f"[CHECKPOINT] Checkpoint deleted : {cera_checkpoint_path}")
else:
    print("[CHECKPOINT] No checkpoint to delete.")